# Data Lakehouse — Pipeline de Datos de Fútbol

Este notebook ejecuta el pipeline completo de integración de datos de jugadores, equipos y ligas a partir de tres fuentes: **Transfermarkt**, **TheSportsDB** y **FBRef**.

**Secciones:**
1. [Ingesta](#1.-Ingesta) — Conexión a S3, inicialización de Spark e ingesta de datos bronce.
2. [ID Matching](#2.-ID-Matching) — Cruce fuzzy de entidades entre fuentes.
3. [Limpieza y Merge](#3.-Limpieza-y-Merge) — Consolidación, validación y escritura de tablas finales.

---
## 1. Ingesta

Instalación de dependencias, conexión al bucket S3 y arranque de la sesión Spark. A continuación se ejecuta la ingesta hacia la capa **bronce** para cada una de las tres fuentes de datos.

### 1.1 Dependencias y conexión a S3

In [3]:
!pip install pyspark
!sudo apt-get update && sudo apt-get install -y openjdk-17-jdk
!python3 -m pip install soccerdata
!java -version
!sudo pip install awscli
!sudo pip install boto3
!sudo pip install rapidfuzz
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://172.16.58.11:31224",
    aws_access_key_id="GK5f421d5f440758f74b0e0312",
    aws_secret_access_key="409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d",
    region_name="us-east-1"
)

# Listar buckets
response = s3.list_buckets()
for bucket in response["Buckets"]:
    print(bucket["Name"])

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:2 https://dl.google.com/linux/chrome-stable/deb stable InRelease [1,825 B] 
Get:3 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,215 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease    
Get:5 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,959 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,090 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Fetched 11.2 MB in 3s (3,633 kB/s)                                             
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.19+10-1~22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 149 not upgraded.
openjdk version "17.0.19" 2026

### 1.2 Inicialización de Spark

In [4]:
# 1. Parar sesión existente
from pyspark.sql import SparkSession
active = SparkSession.getActiveSession()
if active:
    active.stop()

import sys
sys.path.insert(0, '/home/jovyan/work/Data-Lakehouse')
import ingesta
# 3. Crear sesión nueva
spark = ingesta.get_spark()
print (spark.version)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-926e09b7-14f4-4eab-8b3a-a9aa2b71770e;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in spark-list
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in spark-list
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in spark-list
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
:: resolution rep

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
4.1.2


### 1.3 Ingesta por fuente

Se ejecuta `run_ingesta` para cada fuente. Los datos se almacenan en la capa bronce del lakehouse.

In [5]:
spark.sql(f"SHOW NAMESPACES in players").show()

+-----------------+
|        namespace|
+-----------------+
|            fbref|
|      thesportsdb|
|          mockapi|
|concurrency_tests|
|           prueba|
|    transfermarkt|
|          mapping|
|             mock|
|          players|
+-----------------+



In [6]:
spark.sql(f"DROP NAMESPACE players.mock").show()

++
||
++
++



In [7]:
import thesportsdb_bronce
ingesta.run_ingesta(thesportsdb_bronce, test_mode=True, namespace="mock")

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
Getting Big 5 Leagues...
Checking league: English Premier League...
Checking league: German Bundesliga...
Checking league: Italian Serie A...
Checking league: French Ligue 1...
Checking league: Spanish La Liga...
Big 5 Leagues retrieved successfully
Getting Teams...
Getting details for team: Arsenal (ID: 133604)
Getting details for team: Aston Villa (ID: 133601)
Getting details for team: Bournemouth (ID: 134301)
Getting details for team: Bayer Leverkusen (ID: 133666)
Getting details for team: Bayern Munich (ID: 133664)
Getting details for team: Borussia Dortmund (ID: 133650)
Getting details for team: AC Milan (ID: 133667)
Getting details for team: Atalanta (ID: 134782)
Getting details for team: Bologna (ID: 134781)
Getting details for team: Angers (ID: 134709)
Getting details for team: Auxerre (ID: 134788)
Getting details for team: Brest (ID: 133704)
Gett

26/05/28 07:13:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


✅ leagues procesado (CREATE)
🔧 Compactando players.mock.thesportsdb_leagues...
✅ players.mock.thesportsdb_leagues compactado
Processing teams... (15 records [test mode])
✅ teams procesado (CREATE)
🔧 Compactando players.mock.thesportsdb_teams...
✅ players.mock.thesportsdb_teams compactado
Processing players... (75 records [test mode])
✅ players procesado (CREATE)
🔧 Compactando players.mock.thesportsdb_players...
✅ players.mock.thesportsdb_players compactado


{'status': 'ok',
 'tables': [{'table': 'leagues', 'rows': 5, 'action': 'CREATE'},
  {'table': 'teams', 'rows': 15, 'action': 'CREATE'},
  {'table': 'players', 'rows': 75, 'action': 'CREATE'}]}

In [14]:
import transfermarkt_bronce
ingesta.run_ingesta(transfermarkt_bronce, test_mode=True, namespace="mock")

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
Getting Big 5 Leagues...
https://righteous-antonina-overenthusiastically.ngrok-free.dev/competitions/search/Premier%20League
https://righteous-antonina-overenthusiastically.ngrok-free.dev/competitions/search/LaLiga
https://righteous-antonina-overenthusiastically.ngrok-free.dev/competitions/search/Serie%20A
https://righteous-antonina-overenthusiastically.ngrok-free.dev/competitions/search/Bundesliga
https://righteous-antonina-overenthusiastically.ngrok-free.dev/competitions/search/Ligue%201
Getting Teams...
Teams retrieved successfully.
Writing Teams to Iceberg...


Getting Players...


Players retrieved successfully.
Writing Players to Iceberg...
Total player IDs: 75
Andriy Lunin
Victor Lindelöf
Fran González
Ezri Konsa
Éder Militão
Luiz Júnior
Dean Huijsen
Pau Torres
Thibaut Courtois
Diego Conde
Renato Veiga
Logan Costa
Josep Martínez
Renato Marin
Lucas Chevalier
Marco Bizot
Willian Pacho
Emiliano Martínez
Arnau Tenas
Leopold Zingerle
Yann Bisseck
Maarten Vandevoordt
Yann Sommer
Alessandro Bastoni
Joan García
Castello Lukeba
Raffaele Di Gennaro
Péter Gulácsi
Pau Cubarsí
Matvey Safonov
El Chadaille Bitshiabu
Ilya Zabarnyi
Wojciech Szczesny
Diego Kochen
Tommy Setford
William Saliba
David Raya
Eric García
Robin Risser
Gabriel
Kepa Arrizabalaga
Samson Baidoo
Régis Gurtner
Ilan Jourdren
Mathieu Gorgelin
Vanja Milinković-Savić
Alessandro Buongiorno
Mike Maignan
Nikita Contini
Alex Meret
Koni De Winter
Pietro Terracciano
Marc-Aurèle Caillard
Arnaud Bodart
Lorenzo Torriani
Mathias Ferrante
Strahinja Pavlović
Berke Özer
Nathan Ngoy
Alexsandro
Djordje Petrovic
Fraser Forster


Alexander Meyer
Dayot Upamecano
Sven Ulreich
Christos Mandas
Leon Klanac
Processing leagues... (5 records [test mode])
  ➕ Nueva columna detectada: players (bigint)
  ➕ Nueva columna detectada: name (string)
  ➕ Nueva columna detectada: totalMarketValue (bigint)
  ➕ Nueva columna detectada: country (string)
  ➕ Nueva columna detectada: continent (string)
  ➕ Nueva columna detectada: clubs (bigint)
  ➕ Nueva columna detectada: meanMarketValue (bigint)
  ➕ Nueva columna detectada: id (string)
✅ leagues procesado (MERGE)
🔧 Compactando players.mock.transfermarkt_leagues...
✅ players.mock.transfermarkt_leagues compactado
Processing teams... (15 records [test mode])
  ➕ Nueva columna detectada: addressLine2 (string)
  ➕ Nueva columna detectada: league (struct<countryId:string,countryName:string,id:string,name:string,tier:string>)
  ➕ Nueva columna detectada: historicalCrests (array<string>)
  ➕ Nueva columna detectada: foundedOn (string)
  ➕ Nueva columna detectada: url (string)
  ➕ Nueva co

{'status': 'ok',
 'tables': [{'table': 'leagues', 'rows': 5, 'action': 'MERGE'},
  {'table': 'teams', 'rows': 15, 'action': 'MERGE'},
  {'table': 'players', 'rows': 75, 'action': 'MERGE'}]}

In [6]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--user", "soccerdata"],
    check=True
)
import fbref_bronce

fbref_bronce.get_data(namespace="mock", test_mode=True)

[05/28/26 08:26:15] INFO     Saving cached data to /home/jovyan/soccerdata/data/FBref                ]8;id=8466962;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8466963;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\



*** chromedriver to download = 148.0.7778.178 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/148.0.7778.178/linux64/chromedriver-linux64.zip ...
Download Complete!

Extracting ['chromedriver'] from chromedriver-linux64.zip ...
Unzip Complete!

The file [uc_driver] was saved to:
/home/jovyan/.local/lib/python3.11/site-packages/seleniumbase/drivers/
uc_driver

Making [uc_driver 148.0.7778.178] executable ...
[uc_driver 148.0.7778.178] is now ready for use!


*** chromedriver to download = 148.0.7778.178 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/148.0.7778.178/linux64/chromedriver-linux64.zip ...
Download Complete!

Extracting ['chromedriver'] from chromedriver-linux64.zip ...
Unzip Complete!

The file [chromedriver] was saved to:
/home/jovyan/.local/lib/python3.11/site-packages/seleniumbase/drivers/
chromedriver

Making [chromedriver 148.0.7778.178] executable ...
[chromedriver 148.0.7778.178] is now ready for use!



[05/28/26 08:26:17] INFO     No seasons provided. Will retrieve data for the last 5 seasons.         ]8;id=8466968;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8466969;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py#462\462]8;;\

[05/28/26 08:26:42] ERROR    Error while scraping https://fbref.com/en/comps/. Retrying in 0         ]8;id=8466974;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8466975;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py#623\623]8;;\
                             seconds... (attempt 1 of 5).                                                          
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py               
                             ", line 607, in _download_and_save                                                    
                                 response = self._validate_page(url).encode("utf-8")                               
                                            ^^^^^^^^^^^^^^^^^^^^^^^^                                               
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/fbref.py",               
                             line 1031, in _validate_page                                                          
                                 raise Exception(                                                                  
                             Exception: Could not retrieve page content within timeout. Possible                   
                             reasons: failed CAPTCHA, IP block or network issues.                                  

[05/28/26 08:27:08] ERROR    Error while scraping https://fbref.com/en/comps/. Retrying in 10        ]8;id=8466980;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8466981;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py#623\623]8;;\
                             seconds... (attempt 2 of 5).                                                          
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py               
                             ", line 607, in _download_and_save                                                    
                                 response = self._validate_page(url).encode("utf-8")                               
                                            ^^^^^^^^^^^^^^^^^^^^^^^^                                               
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/fbref.py",               
                             line 1031, in _validate_page                                                          
                                 raise Exception(                                                                  
                             Exception: Could not retrieve page content within timeout. Possible                   
                             reasons: failed CAPTCHA, IP block or network issues.                                  

[05/28/26 08:27:46] ERROR    Error while scraping https://fbref.com/en/comps/. Retrying in 20        ]8;id=8466986;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8466987;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py#623\623]8;;\
                             seconds... (attempt 3 of 5).                                                          
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py               
                             ", line 607, in _download_and_save                                                    
                                 response = self._validate_page(url).encode("utf-8")                               
                                            ^^^^^^^^^^^^^^^^^^^^^^^^                                               
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/fbref.py",               
                             line 1031, in _validate_page                                                          
                                 raise Exception(                                                                  
                             Exception: Could not retrieve page content within timeout. Possible                   
                             reasons: failed CAPTCHA, IP block or network issues.                                  

[05/28/26 08:28:34] ERROR    Error while scraping https://fbref.com/en/comps/. Retrying in 30        ]8;id=8466992;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8466993;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py#623\623]8;;\
                             seconds... (attempt 4 of 5).                                                          
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py               
                             ", line 607, in _download_and_save                                                    
                                 response = self._validate_page(url).encode("utf-8")                               
                                            ^^^^^^^^^^^^^^^^^^^^^^^^                                               
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/fbref.py",               
                             line 1031, in _validate_page                                                          
                                 raise Exception(                                                                  
                             Exception: Could not retrieve page content within timeout. Possible                   
                             reasons: failed CAPTCHA, IP block or network issues.                                  

[05/28/26 08:29:30] ERROR    Error while scraping https://fbref.com/en/comps/. Retrying in 40        ]8;id=8466998;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8466999;file:///home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py#623\623]8;;\
                             seconds... (attempt 5 of 5).                                                          
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/_common.py               
                             ", line 607, in _download_and_save                                                    
                                 response = self._validate_page(url).encode("utf-8")                               
                                            ^^^^^^^^^^^^^^^^^^^^^^^^                                               
                               File                                                                                
                             "/home/jovyan/.local/lib/python3.11/site-packages/soccerdata/fbref.py",               
                             line 1031, in _validate_page                                                          
                                 raise Exception(                                                                  
                             Exception: Could not retrieve page content within timeout. Possible                   
                             reasons: failed CAPTCHA, IP block or network issues.                                  

ConnectionError: Could not download https://fbref.com/en/comps/.

---
## 2. ID Matching

Cruce de identidades entre las tres fuentes mediante similitud fuzzy (`rapidfuzz`). Se generan tablas de mapeo y tablas unificadas para **jugadores**, **equipos** y **ligas**.

El proceso sigue cuatro pasos para cada entidad:
1. `generar_mapeo` — encuentra correspondencias entre fuentes usando umbral de similitud.
2. Guardar el mapeo como tabla Iceberg.
3. `unir_fuentes` — construye la tabla unificada a partir del mapeo.
4. Guardar la tabla unificada.

### 2.1 Jugadores

In [ ]:
import importlib
import pandas as pd
import IDMatching
importlib.reload(IDMatching)

# 1) Cargar a pandas
df_tm = spark.table("players.mock.transfermarkt_players").toPandas()
df_ts = spark.table("players.mock.thesportsdb_players").toPandas()
df_fbref = spark.table("players.mock.fbref_players").toPandas()

# 2) Mapeo en pandas (igual que UI)
mapeo_pd = IDMatching.generar_mapeo_df(
    (df_tm,    "name",      "dateOfBirth", "id",       "tm", IDMatching.clave_fecha_completa, 88),
    (df_ts,    "strPlayer", "dateBorn",    "idPlayer", "ts", IDMatching.clave_fecha_completa, 88),
    (df_fbref, "player",    "born",        None,       "fb", IDMatching.clave_solo_anio,      75),
    umbral=85
)

# 3) Preparar FBref con clave derivada para el join (igual concepto que UI)
df_fbref_join = df_fbref.copy()
df_fbref_join["_clave"] = df_fbref_join.apply(
    lambda r: IDMatching.clave_solo_anio(r.get("player"), r.get("born")),
    axis=1
)

# 4) Unificación en pandas (igual que UI)
tabla_final_pd = IDMatching.unir_fuentes_df(
    mapeo_pd,
    (df_tm,        "id",       "tm"),
    (df_ts,        "idPlayer", "ts"),
    (df_fbref_join,"_clave",   "fb"),
)

# 5) Guardar en Iceberg (Spark solo al final, no entre paso 2 y 4)
spark.createDataFrame(mapeo_pd).writeTo("players.mock.players_mapping").using("iceberg").createOrReplace()
spark.createDataFrame(tabla_final_pd).writeTo("players.mock.players_unified").using("iceberg").createOrReplace()

print("Mapeo:       ", len(mapeo_pd))
print("Tabla final: ", len(tabla_final_pd))
print("Filas tm:    ", len(df_tm))

In [8]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)

df_tm = spark.table("players.mock.transfermarkt_players").toPandas()
df_ts = spark.table("players.mock.thesportsdb_players").toPandas()
df_fbref = spark.table("players.mock.fbref_players").toPandas()

# Paso 1: generar mapeo
mapeo = IDMatching.generar_mapeo(
    (df_tm,    "name",      "dateOfBirth", "id",       "tm", IDMatching.clave_fecha_completa, 88),
    (df_ts,    "strPlayer", "dateBorn",    "idPlayer", "ts", IDMatching.clave_fecha_completa, 88),
    (df_fbref, "player",    "born",        None,       "fb", IDMatching.clave_solo_anio,      75),  # umbral más bajo
)

# Paso 2: guardar mapeo
mapeo \
    .writeTo("players.mock.players_mapping") \
    .using("iceberg").createOrReplace()

df_fbref['_clave'] = df_fbref.apply(
    lambda r: IDMatching.clave_solo_anio(r['player'], r['born']), axis=1
)

# Paso 3: unir datos
tabla_final = IDMatching.unir_fuentes(
    mapeo,
    (df_tm,    "id",       "tm"),
    (df_ts,    "idPlayer", "ts"),
    (df_fbref, "_clave",   "fb"),
)

# Paso 4: guardar tabla final
tabla_final.writeTo("players.mock.players_unified") \
    .using("iceberg").createOrReplace()

print("Mapeo:       ", mapeo.count())
print("Tabla final: ", tabla_final.count())
print("Filas tm:    ", len(df_tm))

Total registros únicos: 181
  Con id_tm: 75 (41.4%)
  Con id_ts: 75 (41.4%)
  Con id_fb: 37 (20.4%)
Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog

Total filas en tabla unificada: 182
  Con datos de tm: 75 (41.2%)
  Con datos de ts: 75 (41.2%)
  Con datos de fb: 38 (20.9%)
Mapeo:        181
Tabla final:  182
Filas tm:     75


### 2.2 Equipos

In [7]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)

# Para teams
df_tm = spark.table("players.mock.transfermarkt_teams").toPandas()
df_ts = spark.table("players.mock.thesportsdb_teams").toPandas()
df_fbref = spark.table("players.mock.fbref_teams").toPandas()

# Paso 1: generar mapeo
mapeo = IDMatching.generar_mapeo(
    (df_tm,    "name",      "foundedOn",         "id",     "tm", IDMatching.clave_equipo_fecha, 88),
    (df_ts,    "strTeam",   "intFormedYear",   "idTeam", "ts", IDMatching.clave_equipo_fecha, 88),
    (df_fbref, "team",     None,          None,     "fb", IDMatching.clave_equipo_solo_anio,      75),
)

# Paso 2: guardar mapeo
mapeo \
    .writeTo("players.mock.teams_mapping") \
    .using("iceberg").createOrReplace()

df_fbref['_clave'] = df_fbref.apply(
    lambda r: IDMatching.clave_equipo_solo_anio(r['team'], None), axis=1
)

# Paso 3: unir datos
tabla_final = IDMatching.unir_fuentes(
    mapeo,
    (df_tm,    "id",     "tm"),
    (df_ts,    "idTeam", "ts"),
    (df_fbref, "_clave", "fb"),
)

# Paso 4: guardar tabla final
tabla_final.writeTo("players.mock.teams_unified") \
    .using("iceberg").createOrReplace()

print("Mapeo:       ", mapeo.count())
print("Tabla final: ", tabla_final.count())
print("Filas tm:    ", len(df_tm))

Total registros únicos: 33
  Con id_tm: 15 (45.5%)
  Con id_ts: 1 (3.0%)
  Con id_fb: 19 (57.6%)
Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog

Total filas en tabla unificada: 47
  Con datos de tm: 15 (31.9%)
  Con datos de ts: 15 (31.9%)
  Con datos de fb: 19 (40.4%)
Mapeo:        33
Tabla final:  47
Filas tm:     15


### 2.3 Ligas

In [6]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)

# Para leagues
df_tm = spark.table("players.mock.transfermarkt_leagues").toPandas()
df_ts = spark.table("players.mock.thesportsdb_leagues").toPandas()

# Paso 1: generar mapeo
mapeo = IDMatching.generar_mapeo(
    (df_tm,    "name",      "country",         "id",     "tm", IDMatching.clave_liga, 88),
    (df_ts,    "strLeague",   "strCountry",   "idLeague", "ts", IDMatching.clave_liga, 88),
)

# Paso 2: guardar mapeo
mapeo \
    .writeTo("players.mock.leagues_mapping") \
    .using("iceberg").createOrReplace()

# Paso 3: unir datos
tabla_final = IDMatching.unir_fuentes(
    mapeo,
    (df_tm,    "id",     "tm"),
    (df_ts,    "idLeague", "ts"),
)

# Paso 4: guardar tabla final
tabla_final.writeTo("players.mock.leagues_unified") \
    .using("iceberg").createOrReplace()

print("Mapeo:       ", mapeo.count())
print("Tabla final: ", tabla_final.count())
print("Filas tm:    ", len(df_tm))

Total registros únicos: 5
  Con id_tm: 5 (100.0%)
  Con id_ts: 5 (100.0%)
Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog

Total filas en tabla unificada: 5
  Con datos de tm: 5 (100.0%)
  Con datos de ts: 5 (100.0%)
Mapeo:        5
Tabla final:  5
Filas tm:     5


---
## 3. Limpieza y Merge

A partir de la tabla unificada de jugadores se aplican reglas de limpieza y validación antes de escribir el resultado final en el lakehouse.

- `limpiar_tabla` — resuelve conflictos entre columnas de distintas fuentes aplicando un orden de prioridad (`tiebreaker`).
- `validar_tabla` — detecta registros con problemas y los separa en una tabla de errores.
- Las filas correctas se escriben en `players_merge` y los errores en `players_errores`.

In [9]:
import importlib
import pandas as pd
import subprocess
from pyspark.sql import types as T

subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import plata
importlib.reload(plata)
from plata import limpiar_tabla, validar_tabla, pandas_to_spark

df = spark.table("players.mock.players_unified").toPandas()

grupos = [
    {"col_final": "name", "fuentes": ["tm_name", "ts_strPlayer", "fb_player"], "tiebreaker": "tm_name"},
    {"col_final": "team", "fuentes": ["tm_club_name", "ts_strTeam", "fb_team"], "tiebreaker": "tm_club_name"},
]

df_limpio = limpiar_tabla(df, grupos=grupos)
df_validado, errores = validar_tabla(df_limpio)

# Escribir df_validado
df_validado_spark = pandas_to_spark(spark, df_validado)
df_validado_spark.writeTo("players.mock.players_merge") \
    .using("iceberg").createOrReplace()
print("Tabla merge creada.")

# Escribir errores
errores_spark = pandas_to_spark(spark, errores)
errores_spark.writeTo("players.mock.players_errores") \
    .using("iceberg").createOrReplace()
print("Tabla de errores creada.")

Columnas consolidadas : ['name', 'team']
Total columnas        : 161
Total filas           : 182
Total errores detectados: 109
  ts_intLoved: 75 errores
  ts_intSoccerXMLTeamID: 8 errores
  ts_strNumber: 26 errores
Tabla merge creada.
Tabla de errores creada.


## 4. Consultas

### Top goleadores

In [17]:
spark.sql("""
SELECT
    name,
    team,
    fb_standard_gls
FROM players.mock.players_merge
WHERE fb_standard_gls IS NOT NULL
    AND NOT isnan(fb_standard_gls)
ORDER BY fb_standard_gls DESC
LIMIT 10
""").show()

+-----------+-------+---------------+
|       name|   team|fb_standard_gls|
+-----------+-------+---------------+
|       NULL|   NULL|           13.0|
|       NULL|   NULL|           10.0|
|bukayo saka|arsenal|            5.0|
|       NULL|   NULL|            2.0|
|       NULL|   NULL|            2.0|
|       NULL|   NULL|            2.0|
|       NULL|   NULL|            2.0|
|       NULL|   NULL|            1.0|
|       NULL|   NULL|            1.0|
|       NULL|   NULL|            1.0|
+-----------+-------+---------------+



### Jugadores con más contribuciones (goles + asistencias)

In [22]:
spark.sql("""
SELECT
    name,
    team,
    fb_standard_gls,
    fb_performance_ast,
    (fb_standard_gls + fb_performance_ast) AS contributions
FROM players.mock.players_merge
WHERE fb_standard_gls IS NOT NULL
    AND NOT isnan(fb_standard_gls)
    AND fb_performance_ast IS NOT NULL
    AND NOT isnan(fb_performance_ast)
ORDER BY contributions DESC
LIMIT 10
""").show()

+-----------+-------+---------------+------------------+-------------+
|       name|   team|fb_standard_gls|fb_performance_ast|contributions|
+-----------+-------+---------------+------------------+-------------+
|       NULL|   NULL|           13.0|               2.0|         15.0|
|       NULL|   NULL|           10.0|               1.0|         11.0|
|bukayo saka|arsenal|            5.0|               3.0|          8.0|
|       NULL|   NULL|            2.0|               4.0|          6.0|
|       NULL|   NULL|            1.0|               3.0|          4.0|
|       NULL|   NULL|            0.0|               3.0|          3.0|
|       NULL|   NULL|            2.0|               1.0|          3.0|
|       NULL|   NULL|            2.0|               1.0|          3.0|
|       NULL|   NULL|            1.0|               2.0|          3.0|
|       NULL|   NULL|            1.0|               2.0|          3.0|
+-----------+-------+---------------+------------------+-------------+



### Más goles por 90 minutos

In [25]:
spark.sql("""
SELECT
    name,
    team,
    fb_per_90_minutes_gls
FROM players.mock.players_merge
WHERE fb_per_90_minutes_gls IS NOT NULL
    AND NOT isnan(fb_per_90_minutes_gls)
    AND fb_playing_time_90s IS NOT NULL
    AND NOT isnan(fb_playing_time_90s)
    AND fb_playing_time_90s > 5
ORDER BY fb_per_90_minutes_gls DESC
LIMIT 10
""").show()

+-----------+-------+---------------------+
|       name|   team|fb_per_90_minutes_gls|
+-----------+-------+---------------------+
|       NULL|   NULL|                 0.61|
|       NULL|   NULL|                 0.56|
|       NULL|   NULL|                 0.31|
|bukayo saka|arsenal|                 0.18|
|       NULL|   NULL|                 0.12|
|       NULL|   NULL|                  0.1|
|       NULL|   NULL|                 0.09|
|       NULL|   NULL|                 0.06|
|       NULL|   NULL|                 0.06|
|       NULL|   NULL|                 0.04|
+-----------+-------+---------------------+



### Nacionalidad mas común

In [26]:
spark.sql("""
SELECT
    ts_strNationality,
    COUNT(*) AS total_players
FROM players.mock.players_merge
WHERE ts_strNationality IS NOT NULL
    AND ts_strNationality != ''
GROUP BY ts_strNationality
ORDER BY total_players DESC
LIMIT 10
""").show()

+-----------------+-------------+
|ts_strNationality|total_players|
+-----------------+-------------+
|            spain|           15|
|           france|           15|
|          england|            6|
|            italy|            4|
|           brazil|            3|
|          germany|            3|
|          belgium|            3|
|      ivory coast|            3|
|          denmark|            2|
|           norway|            2|
+-----------------+-------------+



### Jugadores con redes sociales

In [30]:
spark.sql("""
SELECT
    name,
    ts_strInstagram,
    ts_strTwitter
FROM players.mock.players_merge
WHERE (ts_strInstagram IS NOT NULL AND ts_strInstagram != '')
    OR (ts_strTwitter IS NOT NULL AND ts_strTwitter != '')
LIMIT 20
""").show(truncate=False)

+-----------+--------------------------------------+-------------------------------+
|name       |ts_strInstagram                       |ts_strTwitter                  |
+-----------+--------------------------------------+-------------------------------+
|eric garcia|www.instagram.com/ericgm3/            |twitter.com/ericgm3            |
|NULL       |www.instagram.com/ben_white6/         |twitter.com/ben6white          |
|bukayo saka|www.instagram.com/bukayosaka87/       |twitter.com/bukayosaka87       |
|NULL       |www.instagram.com/cnoergaard19/       |                               |
|NULL       |www.instagram.com/its_onana/          |                               |
|NULL       |www.instagram.com/bouba_darkoss/      |twitter.com/boubakamara_4      |
|NULL       |www.instagram.com/a.truffert/         |twitter.com/floriantruffer1    |
|NULL       |www.instagram.com/grimaldo35/         |twitter.com/grimaldo35         |
|NULL       |www.instagram.com/alphonsodaviess/    |twitter.com/a